In [1]:
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, train_test_split
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.load_data_BCICIV import load_all_subjects
from src.preprocess import laplacian_filter
from src.preprocess import normalize_trial
from src.train_EEGNet import train_model, evaluate_model
from models.EEGNet import EEGNet
from torch.utils.data import DataLoader, Dataset, Subset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)


# 22 channels (raw + normalization)

In [2]:
data_path = '../datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path, channels_to_use='all')

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A01T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A02T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: ../datasets/BCICIV_2a_gdf/A07T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A08T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A09T.gdf


In [3]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [4]:
split_index_test = 1150
split_index_val = 862 

dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [5]:
def init_weights_xavier(m):
    if isinstance(m, nn.Conv2d):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
            
    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [6]:
model = EEGNet(22, 1000, 2, f1=16, D=2, dropout_rate=0.3)
model.apply(init_weights_xavier)

trained_model_22_raw = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7202 | Train Acc 0.4838 | Val Loss 0.6937 | Val Acc 0.5000
Epoch 2: Train Loss 0.6954 | Train Acc 0.5394 | Val Loss 0.6949 | Val Acc 0.4896
Epoch 3: Train Loss 0.6925 | Train Acc 0.5186 | Val Loss 0.6925 | Val Acc 0.4896
Epoch 4: Train Loss 0.6741 | Train Acc 0.5708 | Val Loss 0.6916 | Val Acc 0.5069
Epoch 5: Train Loss 0.6684 | Train Acc 0.5800 | Val Loss 0.6882 | Val Acc 0.5174
Epoch 6: Train Loss 0.6610 | Train Acc 0.6067 | Val Loss 0.6836 | Val Acc 0.5174
Epoch 7: Train Loss 0.6569 | Train Acc 0.6218 | Val Loss 0.6828 | Val Acc 0.5278
Epoch 8: Train Loss 0.6417 | Train Acc 0.6253 | Val Loss 0.6845 | Val Acc 0.5104
Epoch 9: Train Loss 0.6404 | Train Acc 0.6439 | Val Loss 0.6817 | Val Acc 0.5347
Epoch 10: Train Loss 0.6407 | Train Acc 0.6473 | Val Loss 0.6779 | Val Acc 0.5451
Epoch 11: Train Loss 0.6243 | Train Acc 0.6729 | Val Loss 0.6747 | Val Acc 0.5590
Epoch 12: Train Loss 0.6213 | Train Acc 0.6647 | Val Loss 0.6674 | Val Acc 0.5660
Epoch 13: Train Loss 0.61

In [ ]:
test_acc = evaluate_model(trained_model_22_raw, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 22 channels + laplacian filter on C3 and C4

In [7]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [7,11] # C3 and C4
        self.c3_neighbours = [1,6,8,13]
        self.c4_neighbours = [5,10,12,17]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = np.expand_dims(x, axis=0)

        x = laplacian_filter(x, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])

        x = x.squeeze(0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [8]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [9]:
model = EEGNet(22, 1000, 2, f1=16, D=2, dropout_rate=0.3)
model.apply(init_weights_he)

trained_model_22_lap = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.6885 | Train Acc 0.5510 | Val Loss 0.6943 | Val Acc 0.5035
Epoch 2: Train Loss 0.6921 | Train Acc 0.5325 | Val Loss 0.6932 | Val Acc 0.5104
Epoch 3: Train Loss 0.6878 | Train Acc 0.5441 | Val Loss 0.6906 | Val Acc 0.5312
Epoch 4: Train Loss 0.6763 | Train Acc 0.5557 | Val Loss 0.6893 | Val Acc 0.5278
Epoch 5: Train Loss 0.6746 | Train Acc 0.5638 | Val Loss 0.6882 | Val Acc 0.5104
Epoch 6: Train Loss 0.6657 | Train Acc 0.6009 | Val Loss 0.6860 | Val Acc 0.5486
Epoch 7: Train Loss 0.6753 | Train Acc 0.5835 | Val Loss 0.6851 | Val Acc 0.5312
Epoch 8: Train Loss 0.6622 | Train Acc 0.6021 | Val Loss 0.6828 | Val Acc 0.5347
Epoch 9: Train Loss 0.6515 | Train Acc 0.6323 | Val Loss 0.6809 | Val Acc 0.5451
Epoch 10: Train Loss 0.6471 | Train Acc 0.6299 | Val Loss 0.6793 | Val Acc 0.5347
Epoch 11: Train Loss 0.6481 | Train Acc 0.6427 | Val Loss 0.6778 | Val Acc 0.5486
Epoch 12: Train Loss 0.6380 | Train Acc 0.6601 | Val Loss 0.6764 | Val Acc 0.5243
Epoch 13: Train Loss 0.64

In [ ]:
test_acc = evaluate_model(trained_model_22_lap, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 22 channels - only mu band

In [10]:
data = load_all_subjects(data_path, use_multiband=True, bands=[(8, 12), (13, 30)], channels_to_use='all')

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/

In [11]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [7,11] # C3 and C4
        self.c3_neighbours = [1,6,8,13]
        self.c4_neighbours = [5,10,12,17]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = x[0]  # Select the mu band (8-12 Hz)

        x = np.expand_dims(x, axis=0)

        x = laplacian_filter(x, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])

        x = x.squeeze(0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [12]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [ ]:
model = EEGNet(22, 1000, 2, f1=16, D=2, dropout_rate=0.3)
model.apply(init_weights_he)

trained_model_22_mu = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.6989 | Train Acc 0.5035 | Val Loss 0.6877 | Val Acc 0.5417
Epoch 2: Train Loss 0.6941 | Train Acc 0.5070 | Val Loss 0.6870 | Val Acc 0.5521
Epoch 3: Train Loss 0.6853 | Train Acc 0.5592 | Val Loss 0.6866 | Val Acc 0.5451
Epoch 4: Train Loss 0.6737 | Train Acc 0.5708 | Val Loss 0.6867 | Val Acc 0.5451
Epoch 5: Train Loss 0.6768 | Train Acc 0.5545 | Val Loss 0.6864 | Val Acc 0.5174
Epoch 6: Train Loss 0.6736 | Train Acc 0.5708 | Val Loss 0.6876 | Val Acc 0.5382
Epoch 7: Train Loss 0.6643 | Train Acc 0.6032 | Val Loss 0.6886 | Val Acc 0.5069
Epoch 8: Train Loss 0.6633 | Train Acc 0.6056 | Val Loss 0.6890 | Val Acc 0.5139
Epoch 9: Train Loss 0.6581 | Train Acc 0.6183 | Val Loss 0.6897 | Val Acc 0.5243
Epoch 10: Train Loss 0.6500 | Train Acc 0.6543 | Val Loss 0.6904 | Val Acc 0.5104
Epoch 11: Train Loss 0.6434 | Train Acc 0.6578 | Val Loss 0.6905 | Val Acc 0.5347
Epoch 12: Train Loss 0.6458 | Train Acc 0.6589 | Val Loss 0.6900 | Val Acc 0.5312
Epoch 13: Train Loss 0.64

In [ ]:
test_acc = evaluate_model(trained_model_22_mu, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 22 channels - mu + beta band

In [14]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [7,11] # C3 and C4
        self.c3_neighbours = [1,6,8,13]
        self.c4_neighbours = [5,10,12,17]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        bands, channels, samples = x.shape

        processed_bands = []
        for band in range(bands):
            band_data = x[band]
            band_data = np.expand_dims(band_data, axis=0)
            band_data = laplacian_filter(band_data, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])
            band_data = band_data.squeeze(0)
            processed_bands.append(band_data)

        x = np.concatenate(processed_bands, axis=0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [15]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [ ]:
model = EEGNet(44, 1000, 2, f1=16, D=2, dropout_rate=0.3)
model.apply(init_weights_he)

trained_model_22_mu_beta = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7080 | Train Acc 0.5035 | Val Loss 0.6900 | Val Acc 0.5660
Epoch 2: Train Loss 0.6969 | Train Acc 0.5058 | Val Loss 0.6869 | Val Acc 0.5625
Epoch 3: Train Loss 0.6894 | Train Acc 0.5278 | Val Loss 0.6841 | Val Acc 0.5625
Epoch 4: Train Loss 0.6839 | Train Acc 0.5650 | Val Loss 0.6825 | Val Acc 0.5833
Epoch 5: Train Loss 0.6814 | Train Acc 0.5615 | Val Loss 0.6806 | Val Acc 0.5694
Epoch 6: Train Loss 0.6816 | Train Acc 0.5534 | Val Loss 0.6797 | Val Acc 0.5417
Epoch 7: Train Loss 0.6729 | Train Acc 0.5858 | Val Loss 0.6788 | Val Acc 0.5556
Epoch 8: Train Loss 0.6655 | Train Acc 0.6148 | Val Loss 0.6773 | Val Acc 0.5660
Epoch 9: Train Loss 0.6621 | Train Acc 0.6125 | Val Loss 0.6761 | Val Acc 0.5451
Epoch 10: Train Loss 0.6637 | Train Acc 0.6137 | Val Loss 0.6750 | Val Acc 0.5556
Epoch 11: Train Loss 0.6531 | Train Acc 0.6439 | Val Loss 0.6735 | Val Acc 0.5486
Epoch 12: Train Loss 0.6515 | Train Acc 0.6276 | Val Loss 0.6731 | Val Acc 0.5625
Epoch 13: Train Loss 0.64

In [ ]:
test_acc = evaluate_model(trained_model_22_mu_beta, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 11 channels (raw + normalization)

In [17]:
data_path = '../datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path)

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A01T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A02T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: ../datasets/BCICIV_2a_gdf/A07T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A08T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A09T.gdf


In [23]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [24]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [25]:
model = EEGNet(11, 1000, 2, f1=16, D=2, dropout_rate=0.3)
model.apply(init_weights_he)

trained_model_11_raw = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7009 | Train Acc 0.5104 | Val Loss 0.6964 | Val Acc 0.4965
Epoch 2: Train Loss 0.6991 | Train Acc 0.5012 | Val Loss 0.6934 | Val Acc 0.4931
Epoch 3: Train Loss 0.6964 | Train Acc 0.5197 | Val Loss 0.6921 | Val Acc 0.5208
Epoch 4: Train Loss 0.6891 | Train Acc 0.5371 | Val Loss 0.6907 | Val Acc 0.5104
Epoch 5: Train Loss 0.6850 | Train Acc 0.5290 | Val Loss 0.6899 | Val Acc 0.5069
Epoch 6: Train Loss 0.6783 | Train Acc 0.5719 | Val Loss 0.6888 | Val Acc 0.5035
Epoch 7: Train Loss 0.6723 | Train Acc 0.5754 | Val Loss 0.6880 | Val Acc 0.5035
Epoch 8: Train Loss 0.6680 | Train Acc 0.5882 | Val Loss 0.6879 | Val Acc 0.5104
Epoch 9: Train Loss 0.6673 | Train Acc 0.5847 | Val Loss 0.6859 | Val Acc 0.5382
Epoch 10: Train Loss 0.6618 | Train Acc 0.6288 | Val Loss 0.6849 | Val Acc 0.5347
Epoch 11: Train Loss 0.6520 | Train Acc 0.6346 | Val Loss 0.6827 | Val Acc 0.5556
Epoch 12: Train Loss 0.6502 | Train Acc 0.6172 | Val Loss 0.6818 | Val Acc 0.5486
Epoch 13: Train Loss 0.64

In [ ]:
test_acc = evaluate_model(trained_model_11_raw, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 11 channels + laplacian filter on C3 and C4

In [26]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [3,7] # C3 and C4
        self.c3_neighbours = [0,2,4,9]
        self.c4_neighbours = [1,6,8,10]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = np.expand_dims(x, axis=0)

        x = laplacian_filter(x, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])

        x = x.squeeze(0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [27]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [28]:
model = EEGNet(11, 1000, 2, f1=16, D=2, dropout_rate=0.3)
model.apply(init_weights_he)

trained_model_11_lap = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.6983 | Train Acc 0.5023 | Val Loss 0.6813 | Val Acc 0.5799
Epoch 2: Train Loss 0.6785 | Train Acc 0.5893 | Val Loss 0.6753 | Val Acc 0.5833
Epoch 3: Train Loss 0.6818 | Train Acc 0.5534 | Val Loss 0.6693 | Val Acc 0.6215
Epoch 4: Train Loss 0.6668 | Train Acc 0.5824 | Val Loss 0.6633 | Val Acc 0.6319
Epoch 5: Train Loss 0.6591 | Train Acc 0.6265 | Val Loss 0.6573 | Val Acc 0.6250
Epoch 6: Train Loss 0.6419 | Train Acc 0.6415 | Val Loss 0.6515 | Val Acc 0.6458
Epoch 7: Train Loss 0.6423 | Train Acc 0.6195 | Val Loss 0.6453 | Val Acc 0.6667
Epoch 8: Train Loss 0.6352 | Train Acc 0.6346 | Val Loss 0.6406 | Val Acc 0.6597
Epoch 9: Train Loss 0.6271 | Train Acc 0.6613 | Val Loss 0.6357 | Val Acc 0.6458
Epoch 10: Train Loss 0.6251 | Train Acc 0.6497 | Val Loss 0.6312 | Val Acc 0.6597
Epoch 11: Train Loss 0.6062 | Train Acc 0.6891 | Val Loss 0.6254 | Val Acc 0.6424
Epoch 12: Train Loss 0.5939 | Train Acc 0.6856 | Val Loss 0.6191 | Val Acc 0.6667
Epoch 13: Train Loss 0.58

In [ ]:
test_acc = evaluate_model(trained_model_11_lap, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 11 channels - only mu band

In [29]:
data = load_all_subjects(data_path, use_multiband=True, bands=[(8, 12), (13, 30)])

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/

In [30]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [3,7] # C3 and C4
        self.c3_neighbours = [0,2,4,9]
        self.c4_neighbours = [1,6,8,10]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = x[0]  # Select the mu band (8-12 Hz)

        x = np.expand_dims(x, axis=0)

        x = laplacian_filter(x, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])

        x = x.squeeze(0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [31]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [32]:
model = EEGNet(11, 1000, 2, f1=16, D=2, dropout_rate=0.3)
model.apply(init_weights_he)

trained_model_11_mu = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7108 | Train Acc 0.4861 | Val Loss 0.6996 | Val Acc 0.5278
Epoch 2: Train Loss 0.6931 | Train Acc 0.5360 | Val Loss 0.6963 | Val Acc 0.5243
Epoch 3: Train Loss 0.6956 | Train Acc 0.5186 | Val Loss 0.6949 | Val Acc 0.5417
Epoch 4: Train Loss 0.6712 | Train Acc 0.5963 | Val Loss 0.6932 | Val Acc 0.5660
Epoch 5: Train Loss 0.6755 | Train Acc 0.5940 | Val Loss 0.6915 | Val Acc 0.5590
Epoch 6: Train Loss 0.6641 | Train Acc 0.6090 | Val Loss 0.6897 | Val Acc 0.5590
Epoch 7: Train Loss 0.6565 | Train Acc 0.6334 | Val Loss 0.6883 | Val Acc 0.5590
Epoch 8: Train Loss 0.6598 | Train Acc 0.6125 | Val Loss 0.6861 | Val Acc 0.5556
Epoch 9: Train Loss 0.6394 | Train Acc 0.6636 | Val Loss 0.6846 | Val Acc 0.5660
Epoch 10: Train Loss 0.6463 | Train Acc 0.6659 | Val Loss 0.6831 | Val Acc 0.5625
Epoch 11: Train Loss 0.6392 | Train Acc 0.6740 | Val Loss 0.6814 | Val Acc 0.5590
Epoch 12: Train Loss 0.6204 | Train Acc 0.6937 | Val Loss 0.6786 | Val Acc 0.5799
Epoch 13: Train Loss 0.62

In [ ]:
test_acc = evaluate_model(trained_model_11_mu, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 11 channels - mu + beta band

In [33]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [3,7] # C3 and C4
        self.c3_neighbours = [0,2,4,9]
        self.c4_neighbours = [1,6,8,10]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        bands, channels, samples = x.shape

        processed_bands = []
        for band in range(bands):
            band_data = x[band]
            band_data = np.expand_dims(band_data, axis=0)
            band_data = laplacian_filter(band_data, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])
            band_data = band_data.squeeze(0)
            processed_bands.append(band_data)

        x = np.concatenate(processed_bands, axis=0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [34]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [35]:
model = EEGNet(22, 1000, 2, f1=16, D=2, dropout_rate=0.3)
model.apply(init_weights_he)

trained_model_11_mu_beta = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7033 | Train Acc 0.4814 | Val Loss 0.6995 | Val Acc 0.4826
Epoch 2: Train Loss 0.6932 | Train Acc 0.5232 | Val Loss 0.6964 | Val Acc 0.5035
Epoch 3: Train Loss 0.6901 | Train Acc 0.5534 | Val Loss 0.6943 | Val Acc 0.5069
Epoch 4: Train Loss 0.6830 | Train Acc 0.5441 | Val Loss 0.6921 | Val Acc 0.5243
Epoch 5: Train Loss 0.6760 | Train Acc 0.5905 | Val Loss 0.6899 | Val Acc 0.5278
Epoch 6: Train Loss 0.6645 | Train Acc 0.6148 | Val Loss 0.6875 | Val Acc 0.5451
Epoch 7: Train Loss 0.6704 | Train Acc 0.5905 | Val Loss 0.6857 | Val Acc 0.5382
Epoch 8: Train Loss 0.6657 | Train Acc 0.6056 | Val Loss 0.6832 | Val Acc 0.5451
Epoch 9: Train Loss 0.6629 | Train Acc 0.6172 | Val Loss 0.6825 | Val Acc 0.5382
Epoch 10: Train Loss 0.6545 | Train Acc 0.6230 | Val Loss 0.6802 | Val Acc 0.5312
Epoch 11: Train Loss 0.6474 | Train Acc 0.6613 | Val Loss 0.6785 | Val Acc 0.5451
Epoch 12: Train Loss 0.6377 | Train Acc 0.6589 | Val Loss 0.6772 | Val Acc 0.5694
Epoch 13: Train Loss 0.64

In [ ]:
test_acc = evaluate_model(trained_model_11_mu_beta, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')